# Iliad API Image Processing Test

This notebook tests the Iliad API image processing capabilities:
1. **Image Analysis via Multimodal Chat** - Get detailed descriptions and analysis of images using vision-capable LLMs
2. OCR text extraction from images via `/api/v1/recognize/ocr` (legacy)
3. Document recognition via `/api/v1/recognize`
4. **Processing attachments from Confluence pages by ID**

The primary method for image processing is now the **multimodal chat endpoint** which provides:
- Detailed descriptions of visual content
- Understanding of diagrams, charts, and architecture images
- Context-aware analysis of images
- More comprehensive output than simple OCR

Use this notebook to debug and verify that image processing is working correctly.

In [1]:
import sys
sys.path.append('..')

from pathlib import Path
from src.config import ConfigConfluenceRag
from src.iliad.client import IliadClient, IliadClientConfig
from src.iliad.recognize import TextRecognizer
from src.confluence.rest_client import ConfluenceRestClient
from src.preprocessing.attachment_fetcher import AttachmentFetcher
from loguru import logger
import requests

# Configure logging for detailed output
logger.remove()
logger.add(sys.stderr, level="DEBUG")

1

## 1. Validate Configuration

In [2]:
# Check that Iliad API configuration is present
print("Checking Iliad API configuration...")
print(f"ILIAD_API_URL: {ConfigConfluenceRag.ILIAD_API_URL}")
print(f"ILIAD_API_KEY: {'*' * 8 + ConfigConfluenceRag.ILIAD_API_KEY[-4:] if ConfigConfluenceRag.ILIAD_API_KEY else 'NOT SET'}")

if not ConfigConfluenceRag.ILIAD_API_URL:
    raise ValueError("ILIAD_API_URL is not set in .env")
if not ConfigConfluenceRag.ILIAD_API_KEY:
    raise ValueError("ILIAD_API_KEY is not set in .env")

print("\n✅ Iliad configuration looks good")

# Check Confluence configuration
print("\nChecking Confluence configuration...")
print(f"CONFLUENCE_URL: {ConfigConfluenceRag.CONFLUENCE_URL}")
print(f"CONFLUENCE_USERNAME: {ConfigConfluenceRag.CONFLUENCE_USERNAME}")
print(f"CONFLUENCE_API_TOKEN: {'*' * 8 + ConfigConfluenceRag.CONFLUENCE_API_TOKEN[-4:] if ConfigConfluenceRag.CONFLUENCE_API_TOKEN else 'NOT SET'}")

if not ConfigConfluenceRag.CONFLUENCE_URL:
    print("⚠️  CONFLUENCE_URL is not set - Confluence tests will be skipped")
else:
    print("\n✅ Confluence configuration looks good")

Checking Iliad API configuration...
ILIAD_API_URL: https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/chat/gpt-4o
ILIAD_API_KEY: ********cvC9

✅ Iliad configuration looks good

Checking Confluence configuration...
CONFLUENCE_URL: https://confluence.abbvienet.com/
CONFLUENCE_USERNAME: garrick.bruening@abbvie.com
CONFLUENCE_API_TOKEN: ********qt4w

✅ Confluence configuration looks good


## 2. Initialize Iliad Client

In [3]:
# Initialize the Iliad client from environment variables
print("Initializing IliadClient...")

try:
    config = IliadClientConfig.from_env()
    client = IliadClient(config)
    print(f"\n✅ IliadClient initialized")
    print(f"   Base URL: {config.base_url}")
    print(f"   Timeout: {config.timeout}s")
    print(f"   Max retries: {config.max_retries}")
except Exception as e:
    print(f"\n❌ Failed to initialize IliadClient: {e}")
    raise

2026-03-06 10:56:42.282 | INFO     | src.iliad.client:__init__:135 - Initialized IliadClient with base URL: https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/chat/gpt-4o


Initializing IliadClient...

✅ IliadClient initialized
   Base URL: https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/chat/gpt-4o
   Timeout: 120s
   Max retries: 3


## 3. Initialize TextRecognizer

In [4]:
# Initialize the TextRecognizer which provides high-level text extraction
print("Initializing TextRecognizer...")

recognizer = TextRecognizer(client)

# Display supported file types
print("\nSupported file extensions:")
extensions = recognizer.get_supported_extensions()
for category, exts in extensions.items():
    print(f"  {category}: {', '.join(exts)}")

2026-03-06 10:56:42.807 | INFO     | src.iliad.recognize:__init__:88 - Initialized TextRecognizer


Initializing TextRecognizer...

Supported file extensions:
  document: .doc, .docx, .odg, .odp, .ods, .odt, .pdf, .ppt, .pptx, .rtf, .xls, .xlsm, .xlsx
  text: .csv, .eml, .epub, .html, .json, .md, .tsv, .txt, .xhtml, .xml
  image: .jpeg, .jpg, .png, .tif, .tiff
  audio: .amr, .flac, .m4a, .mp3, .mp4, .ogg, .wav, .webm


---
# Part A: Test with Confluence Page by ID

This section allows you to fetch a specific Confluence page by its ID and process any image attachments.

## A1. Initialize Confluence Client

In [5]:
# Initialize Confluence client
confluence_client = None

if ConfigConfluenceRag.CONFLUENCE_URL:
    print("Initializing Confluence client...")
    
    # Try Bearer token authentication first
    print("Attempting Bearer token authentication...")
    confluence_client = ConfluenceRestClient(
        base_url=ConfigConfluenceRag.CONFLUENCE_URL,
        username=ConfigConfluenceRag.CONFLUENCE_USERNAME,
        api_token=ConfigConfluenceRag.CONFLUENCE_API_TOKEN,
        verify_ssl=False,
        auth_type='bearer'
    )
    
    if not confluence_client.test_connection():
        print("\nBearer token failed. Trying Basic authentication...")
        confluence_client = ConfluenceRestClient(
            base_url=ConfigConfluenceRag.CONFLUENCE_URL,
            username=ConfigConfluenceRag.CONFLUENCE_USERNAME,
            api_token=ConfigConfluenceRag.CONFLUENCE_API_TOKEN,
            verify_ssl=False,
            auth_type='basic'
        )
        if not confluence_client.test_connection():
            print("\n❌ Authentication failed with both Bearer and Basic auth methods")
            confluence_client = None
        else:
            print("\n✅ Confluence client initialized with Basic auth")
    else:
        print("\n✅ Confluence client initialized with Bearer auth")
else:
    print("⚠️  Confluence not configured - skipping Confluence client initialization")

Initializing Confluence client...
Attempting Bearer token authentication...
Testing connection to: https://confluence.abbvienet.com/rest/api/space
Auth type: bearer
Response status: 200
✓ Connection successful!

✅ Confluence client initialized with Bearer auth


## A2. Fetch Confluence Page by ID

Enter your Confluence page ID below. You can find the page ID in the URL when viewing the page, or use the Confluence REST API.

In [6]:
# ============================================
# SET YOUR CONFLUENCE PAGE ID HERE
# ============================================
CONFLUENCE_PAGE_ID = "330213695"  # e.g., "123456789"
# ============================================

if not CONFLUENCE_PAGE_ID:
    print("⚠️  Please set CONFLUENCE_PAGE_ID above to test with a Confluence page")
    print("   You can find the page ID in the Confluence page URL")
    print("   Example: https://confluence.example.com/pages/viewpage.action?pageId=123456789")
else:
    print(f"Will test with Confluence page ID: {CONFLUENCE_PAGE_ID}")

Will test with Confluence page ID: 330213695


In [7]:
# Fetch the Confluence page
confluence_page = None

if confluence_client and CONFLUENCE_PAGE_ID:
    print(f"Fetching Confluence page: {CONFLUENCE_PAGE_ID}")
    print("-" * 60)
    
    try:
        confluence_page = confluence_client.get_page_by_id(CONFLUENCE_PAGE_ID)
        
        if confluence_page:
            print(f"\n✅ Page fetched successfully!")
            print(f"   Title: {confluence_page.title}")
            print(f"   URL: {confluence_page.url}")
            print(f"   Space: {confluence_page.space_key} - {confluence_page.space_name}")
            print(f"   Author: {confluence_page.author}")
            print(f"   Version: {confluence_page.version}")
            print(f"   Modified: {confluence_page.modified_date}")
            print(f"   Content length: {len(confluence_page.content_text or '')} characters")
            if confluence_page.parent_title:
                print(f"   Parent: {confluence_page.parent_title}")
        else:
            print(f"\n❌ Page not found or access denied")
            
    except Exception as e:
        print(f"\n❌ Error fetching page: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
else:
    if not confluence_client:
        print("⚠️  Confluence client not initialized")
    if not CONFLUENCE_PAGE_ID:
        print("⚠️  No page ID specified")

Fetching Confluence page: 330213695
------------------------------------------------------------

✅ Page fetched successfully!
   Title: Confluence Architecture
   URL: https://confluence.abbvienet.com/spaces/DSA/pages/330213695/Confluence+Architecture
   Space: DSA - Data Science and Analytics
   Author: Bruening, Gary W
   Version: 3
   Modified: 2026-03-02T07:47:53.204-08:00
   Content length: 1522 characters
   Parent: Confluence RAG Agent


## A3. List Page Attachments

In [8]:
# Fetch attachments for the page
attachments = []

if confluence_client and CONFLUENCE_PAGE_ID:
    print(f"Fetching attachments for page: {CONFLUENCE_PAGE_ID}")
    print("-" * 60)
    
    try:
        attachments = confluence_client.get_attachments(CONFLUENCE_PAGE_ID)
        
        if attachments:
            print(f"\n✅ Found {len(attachments)} attachment(s):")
            print()
            
            for i, att in enumerate(attachments, 1):
                file_size_kb = att.get('fileSize', 0) / 1024
                print(f"  {i}. {att.get('title', 'Unknown')}")
                print(f"     Type: {att.get('mediaType', 'Unknown')}")
                print(f"     Size: {file_size_kb:.1f} KB")
                print(f"     ID: {att.get('id', 'Unknown')}")
                print(f"     Download URL: {att.get('download_url', 'N/A')[:80]}...")
                print()
        else:
            print("\n⚠️  No attachments found on this page")
            
    except Exception as e:
        print(f"\n❌ Error fetching attachments: {type(e).__name__}: {e}")
else:
    print("⚠️  Cannot fetch attachments - client or page ID not configured")

2026-03-06 10:56:48.701 | DEBUG    | src.confluence.rest_client:get_attachments:419 - Found 7 attachments for page 330213695


Fetching attachments for page: 330213695
------------------------------------------------------------

✅ Found 7 attachment(s):

  1. Confluence RAG Agent.png
     Type: image/png
     Size: 784.1 KB
     ID: 330214378
     Download URL: https://confluence.abbvienet.com/download/attachments/330213695/Confluence%20RAG...

  2. Confluence RAG Agent
     Type: application/vnd.jgraph.mxfile
     Size: 25.6 KB
     ID: 330214377
     Download URL: https://confluence.abbvienet.com/download/attachments/330213695/Confluence%20RAG...

  3. ~Confluence RAG Agent.tmp
     Type: application/x-drawio-draft
     Size: 25.6 KB
     ID: 330214375
     Download URL: https://confluence.abbvienet.com/download/attachments/330213695/~Confluence%20RA...

  4. ~Confluence_rag_flowchart.tmp
     Type: application/x-drawio-draft
     Size: 20.8 KB
     ID: 330213706
     Download URL: https://confluence.abbvienet.com/download/attachments/330213695/~Confluence_rag_...

  5. Confluence_rag_flowchart.png
     Typ

## A4. Filter Image Attachments

In [9]:
# Filter to only image attachments
IMAGE_MIME_TYPES = {'image/png', 'image/jpeg', 'image/jpg', 'image/tiff', 'image/gif', 'image/bmp'}
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.tiff', '.tif', '.gif', '.bmp'}

image_attachments = []

if attachments:
    for att in attachments:
        media_type = att.get('mediaType', '').lower()
        filename = att.get('title', '').lower()
        ext = Path(filename).suffix.lower()
        
        if media_type in IMAGE_MIME_TYPES or ext in IMAGE_EXTENSIONS:
            image_attachments.append(att)
    
    print(f"Found {len(image_attachments)} image attachment(s) out of {len(attachments)} total:")
    for att in image_attachments:
        print(f"  - {att.get('title')} ({att.get('mediaType')})")
else:
    print("No attachments to filter")

Found 3 image attachment(s) out of 7 total:
  - Confluence RAG Agent.png (image/png)
  - Confluence_rag_flowchart.png (image/png)
  - Architecture.drawio.png (image/png)


## A5. Download and Process Image Attachments (Multimodal Analysis)

This section uses the **multimodal chat endpoint** to analyze images. Unlike OCR which only extracts text, this method:
- Provides detailed descriptions of visual content
- Understands diagrams, flowcharts, and architecture images
- Describes relationships and context within images

In [10]:
# Create directory for downloaded attachments
attachment_dir = Path("test_files") / "confluence_attachments"
attachment_dir.mkdir(parents=True, exist_ok=True)

print(f"Attachment download directory: {attachment_dir.absolute()}")

Attachment download directory: /Users/bruengw/Library/CloudStorage/OneDrive-AbbVieInc(O365)/Documents/confluence_rag/notebooks/test_files/confluence_attachments


In [11]:
# Download and process each image attachment using multimodal image analysis
processed_results = []

if image_attachments and confluence_client:
    print(f"Processing {len(image_attachments)} image attachment(s) with multimodal analysis...")
    print("=" * 60)
    
    for i, att in enumerate(image_attachments, 1):
        filename = att.get('title', f'attachment_{i}')
        download_url = att.get('download_url', '')
        
        print(f"\n[{i}/{len(image_attachments)}] Processing: {filename}")
        print("-" * 40)
        
        result = {
            'filename': filename,
            'attachment_id': att.get('id'),
            'download_url': download_url,
            'downloaded': False,
            'analysis_success': False,
            'extracted_text': '',
            'error': None
        }
        
        if not download_url:
            print("  ⚠️  No download URL available")
            result['error'] = 'No download URL'
            processed_results.append(result)
            continue
        
        # Download the attachment
        local_path = attachment_dir / filename
        print(f"  Downloading to: {local_path}")
        
        try:
            content = confluence_client.download_attachment(download_url)
            
            if content:
                with open(local_path, 'wb') as f:
                    f.write(content)
                result['downloaded'] = True
                print(f"  ✅ Downloaded: {len(content)} bytes")
                
                # Process with multimodal image analysis (NOT OCR)
                print(f"  Running multimodal image analysis...")
                try:
                    extracted_text = client.analyze_image(str(local_path))
                    result['extracted_text'] = extracted_text
                    result['analysis_success'] = True
                    
                    if extracted_text:
                        print(f"  ✅ Analysis extracted {len(extracted_text)} characters")
                        preview = extracted_text[:300].replace('\n', ' ')
                        print(f"  Preview: {preview}..." if len(extracted_text) > 300 else f"  Content: {preview}")
                    else:
                        print(f"  ⚠️  Analysis returned empty text")
                        
                except Exception as e:
                    print(f"  ❌ Analysis Error: {type(e).__name__}: {e}")
                    result['error'] = f'Analysis Error: {e}'
            else:
                print(f"  ❌ Download returned empty content")
                result['error'] = 'Download returned empty'
                
        except Exception as e:
            print(f"  ❌ Download Error: {type(e).__name__}: {e}")
            result['error'] = f'Download Error: {e}'
        
        processed_results.append(result)
    
    print("\n" + "=" * 60)
    print("Processing complete!")
else:
    if not image_attachments:
        print("No image attachments to process")
    else:
        print("Confluence client not available")

2026-03-06 10:57:01.559 | DEBUG    | src.confluence.rest_client:download_attachment:459 - Trying to download from: https://confluence.abbvienet.com/download/attachments/330213695/Confluence%20RAG%20Agent.png?version=10&modificationDate=1772466443707&api=v2


Processing 3 image attachment(s) with multimodal analysis...

[1/3] Processing: Confluence RAG Agent.png
----------------------------------------


2026-03-06 10:57:02.276 | INFO     | src.iliad.client:analyze_image:544 - Analyzing image with model gpt-4o
2026-03-06 10:57:02.277 | DEBUG    | src.iliad.client:_make_request:190 - Request attempt 1/3: POST https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/chat/gpt-4o


  ✅ Downloaded: 802883 bytes
  Running multimodal image analysis...


2026-03-06 10:57:14.720 | DEBUG    | src.confluence.rest_client:download_attachment:459 - Trying to download from: https://confluence.abbvienet.com/download/attachments/330213695/Confluence_rag_flowchart.png?version=1&modificationDate=1772121124872&api=v2


...

  ✅ Downloaded: 155790 bytes
  Running multimodal image analysis...


2026-03-06 10:57:24.385 | DEBUG    | src.confluence.rest_client:download_attachment:459 - Trying to download from: https://confluence.abbvienet.com/download/attachments/330213695/Architecture.drawio.png?version=1&modificationDate=1772120647531&api=v2


  ✅ Analysis extracted 1834 characters
  Preview: This image is a flowchart illustrating an information retrieval system using a Retrieving Augmented Generation (RAG) pipeline. It consists of several interconnected layers:  ### 1. **Data Preparation Layer**  - **Data Acquisition Layer:**   - **REST API Client**: Fetches pages from Confluence using ...

[3/3] Processing: Architecture.drawio.png
----------------------------------------


2026-03-06 10:57:24.904 | INFO     | src.iliad.client:analyze_image:544 - Analyzing image with model gpt-4o
2026-03-06 10:57:24.904 | DEBUG    | src.iliad.client:_make_request:190 - Request attempt 1/3: POST https://api-epic.ir-gateway.abbvienet.com/iliad/api/v1/chat/gpt-4o


  ✅ Downloaded: 189874 bytes
  Running multimodal image analysis...
  ✅ Analysis extracted 2416 characters
  Preview: The image is a flowchart illustrating a data processing pipeline for a Retrieval-Augmented Generation (RAG) system. It is divided into various sections, each representing a different part of the system. Here is a detailed breakdown:  ### 1. Data Preparation Layer - **Data Acquisition Layer**:    - *...

Processing complete!


## A6. Summary of Processed Attachments

In [12]:
# Display summary of all processed attachments
if processed_results:
    print("Processing Summary")
    print("=" * 60)
    
    successful = sum(1 for r in processed_results if r['analysis_success'])
    failed = len(processed_results) - successful
    
    print(f"Total attachments processed: {len(processed_results)}")
    print(f"Successful image analysis: {successful}")
    print(f"Failed: {failed}")
    print()
    
    for i, result in enumerate(processed_results, 1):
        status = "✅" if result['analysis_success'] else "❌"
        text_len = len(result['extracted_text'])
        print(f"{status} {i}. {result['filename']}")
        print(f"      Downloaded: {result['downloaded']}")
        print(f"      Analysis Success: {result['analysis_success']}")
        print(f"      Text Length: {text_len} chars")
        if result['error']:
            print(f"      Error: {result['error']}")
        print()
else:
    print("No results to display")

Processing Summary
Total attachments processed: 3
Successful image analysis: 3
Failed: 0

✅ 1. Confluence RAG Agent.png
      Downloaded: True
      Analysis Success: True
      Text Length: 2434 chars

✅ 2. Confluence_rag_flowchart.png
      Downloaded: True
      Analysis Success: True
      Text Length: 1834 chars

✅ 3. Architecture.drawio.png
      Downloaded: True
      Analysis Success: True
      Text Length: 2416 chars



## A7. View Full Extracted Text

In [15]:
# Display full extracted text for each successful result
if processed_results:
    for i, result in enumerate(processed_results, 1):
        if result['analysis_success'] and result['extracted_text']:
            print(f"\n{'=' * 60}")
            print(f"Full OCR Text for: {result['filename']}")
            print(f"{'=' * 60}")
            print(result['extracted_text'])
            print()
else:
    print("No results available")


Full OCR Text for: Confluence RAG Agent.png
The image is a detailed flowchart illustrating a system architecture involving data preparation, processing, and user interaction. Here's a summary of the main components and their interactions:

### Layers and Components:

1. **Data Preparation Layer:**
   - **Data Acquisition Layer:**
     - Fetches pages from Confluence using a REST API Client (`src/confluence/rest_client.py`).
     - Parses the HTML content using an HTML Parser (`src/confluence/parser.py`).
     - Saves raw page data and metadata in `confluence_pages.json`.

   - **Vectorization Layer:**
     - **Text Chunker:** Splits text into manageable chunks.
     - **Embedding Manager:** Uses `sentence-transformers` (`src/rag/embeddings.py`) to generate embeddings from text chunks.
     - The vectors are stored in a Vector Database (`ChromaDB`, `src/rag/vectorstore.py`).

2. **RAG Pipeline Layer:**
   - **Query Processor:**
     - Uses `src/rag/query_processor.py` with NLTK for tex

## A8. Use AttachmentFetcher (Alternative Method)

This uses the `AttachmentFetcher` class which handles downloading and processing in one step.

In [ ]:
# Alternative: Use AttachmentFetcher for integrated processing
if confluence_client and CONFLUENCE_PAGE_ID:
    print("Using AttachmentFetcher for integrated processing...")
    print("-" * 60)
    
    try:
        fetcher = AttachmentFetcher(
            confluence_client=confluence_client,
            iliad_client=client,
            storage_path="test_files/attachment_fetcher_storage"
        )
        
        # Process all attachments (not just images)
        combined_content = fetcher.process_all_page_attachments(
            page_id=CONFLUENCE_PAGE_ID,
            cleanup=False  # Keep files for inspection
        )
        
        if combined_content:
            print(f"\n✅ AttachmentFetcher extracted {len(combined_content)} characters total")
            print("\nExtracted Content Preview:")
            print("-" * 40)
            print(combined_content[:2000])
            if len(combined_content) > 2000:
                print(f"\n... ({len(combined_content) - 2000} more characters)")
        else:
            print("\n⚠️  No content extracted from attachments")
            
    except Exception as e:
        print(f"\n❌ Error: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  Cannot use AttachmentFetcher - client or page ID not configured")

## A9. Debug: Raw API Response for Confluence Attachment

Test both the legacy OCR endpoint and the new multimodal image analysis endpoint.

In [ ]:
# Debug: Test multimodal image analysis API with a specific downloaded attachment
if processed_results:
    # Find first successfully downloaded file
    for result in processed_results:
        if result['downloaded']:
            test_file = attachment_dir / result['filename']
            if test_file.exists():
                print(f"Testing multimodal image analysis with: {test_file}")
                print("-" * 60)
                
                # Test the new analyze_image method
                print("\n📸 Testing IliadClient.analyze_image() (Multimodal Chat):")
                print("-" * 40)
                try:
                    analysis = client.analyze_image(
                        str(test_file),
                        prompt="Describe this image in detail. What does it show? If it's a diagram or flowchart, explain the components and their relationships."
                    )
                    print(f"Analysis length: {len(analysis)} chars")
                    print(f"\nFull Analysis:\n{analysis}")
                except Exception as e:
                    print(f"Error: {type(e).__name__}: {e}")
                
                # Also test legacy OCR for comparison
                print("\n\n📝 Testing IliadClient.recognize_ocr() (Legacy OCR):")
                print("-" * 40)
                try:
                    ocr_text = client.recognize_ocr(str(test_file))
                    print(f"OCR length: {len(ocr_text)} chars")
                    print(f"\nOCR Text:\n{ocr_text[:1000]}..." if len(ocr_text) > 1000 else f"\nOCR Text:\n{ocr_text}")
                except Exception as e:
                    print(f"Error: {type(e).__name__}: {e}")
                
                break
    else:
        print("No successfully downloaded files found for debugging")
else:
    print("No processed results available")

---
# Part B: Test with Local Files

This section tests OCR with locally created test images.

## B1. Create Test Image

In [ ]:
# Create a test directory for our test files
test_dir = Path("test_files")
test_dir.mkdir(exist_ok=True)

print(f"Test directory: {test_dir.absolute()}")

In [ ]:
# Try to create a test image with PIL (if available)
try:
    from PIL import Image, ImageDraw, ImageFont
    
    # Create a simple image with text
    img = Image.new('RGB', (800, 200), color='white')
    draw = ImageDraw.Draw(img)
    
    # Try to use a basic font, fall back to default if not available
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 36)
    except Exception:
        font = ImageFont.load_default()
    
    # Draw test text
    test_text = "Hello World! This is a test image for OCR."
    draw.text((20, 80), test_text, fill='black', font=font)
    
    # Save the test image
    test_image_path = test_dir / "test_ocr_image.png"
    img.save(test_image_path)
    
    print(f"✅ Created test image: {test_image_path}")
    print(f"   Expected text: '{test_text}'")
    
    # Display the image inline (if in Jupyter)
    display(img)
    
except ImportError:
    print("⚠️  PIL/Pillow not installed. Install with: pip install Pillow")
    print("   You can manually place a test image at: test_files/test_ocr_image.png")
    test_image_path = None

## B2. Test Raw Image Analysis Endpoint (Multimodal Chat)

In [ ]:
# Test multimodal image analysis with test image
import base64

if test_image_path and test_image_path.exists():
    print(f"Testing multimodal image analysis with: {test_image_path}")
    print("-" * 60)
    
    try:
        # Test using IliadClient.analyze_image()
        print("\n📸 Testing IliadClient.analyze_image():")
        analysis = client.analyze_image(
            str(test_image_path),
            prompt="Describe what you see in this image. Include any text that appears."
        )
        print(f"\n✅ Image Analysis Result:")
        print(f"   Length: {len(analysis)} characters")
        print(f"   Content:\n{analysis}")
        
    except Exception as e:
        print(f"\n❌ Request failed: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  No test image available. Please create one first.")

## B3. Compare: OCR vs Image Analysis

Compare the results from legacy OCR (text extraction only) vs multimodal image analysis (detailed descriptions).

In [ ]:
# Compare OCR vs Image Analysis methods
if test_image_path and test_image_path.exists():
    print(f"Comparing methods with: {test_image_path}")
    print("=" * 60)
    
    # Method 1: Legacy OCR
    print("\n📝 Method 1: IliadClient.recognize_ocr() (Legacy OCR)")
    print("-" * 40)
    try:
        ocr_result = client.recognize_ocr(str(test_image_path))
        print(f"   Type: {type(ocr_result)}")
        print(f"   Length: {len(ocr_result)} characters")
        print(f"   Content: '{ocr_result}'")
    except Exception as e:
        print(f"   ❌ Error: {type(e).__name__}: {e}")
        ocr_result = ""
    
    # Method 2: Multimodal Image Analysis
    print("\n\n📸 Method 2: IliadClient.analyze_image() (Multimodal Chat)")
    print("-" * 40)
    try:
        analysis_result = client.analyze_image(str(test_image_path))
        print(f"   Type: {type(analysis_result)}")
        print(f"   Length: {len(analysis_result)} characters")
        print(f"   Content:\n{analysis_result}")
    except Exception as e:
        print(f"   ❌ Error: {type(e).__name__}: {e}")
        analysis_result = ""
    
    # Summary
    print("\n\n📊 Summary Comparison:")
    print("-" * 40)
    print(f"   OCR extracted: {len(ocr_result)} chars (text only)")
    print(f"   Analysis extracted: {len(analysis_result)} chars (detailed description)")
    
    if analysis_result:
        print("\n   ✅ Image Analysis provides more comprehensive output!")
else:
    print("⚠️  No test image available.")

## B4. Test Document Recognition

In [ ]:
# Create a simple text file to test document recognition
txt_path = test_dir / "test_document.txt"
txt_content = """This is a test document.
It contains multiple lines of text.
The purpose is to test the recognize endpoint.
"""

with open(txt_path, 'w') as f:
    f.write(txt_content)

print(f"Created test document: {txt_path}")
print(f"Content:\n{txt_content}")

In [ ]:
# Test document recognition
print("Testing document recognition...")
print("-" * 60)

try:
    result = client.recognize(str(txt_path))
    print(f"\n✅ Document Recognition Result:")
    print(f"   Type: {type(result)}")
    print(f"   Length: {len(result)} characters")
    print(f"   Content:\n{result}")
except Exception as e:
    print(f"\n❌ Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

---
# Cleanup

In [ ]:
# Optional: Clean up test files
cleanup = False  # Set to True to delete test files

if cleanup and test_dir.exists():
    import shutil
    shutil.rmtree(test_dir)
    print(f"✅ Cleaned up test directory: {test_dir}")
else:
    print(f"Test files preserved at: {test_dir.absolute()}")
    if test_dir.exists():
        print(f"\nContents:")
        for item in test_dir.rglob('*'):
            if item.is_file():
                print(f"  {item.relative_to(test_dir)}")

---
# Summary

## Part A: Confluence Page Testing
- Fetch a specific Confluence page by ID
- List all attachments on the page
- Filter for image attachments
- **Download and process each image with multimodal LLM analysis** (provides detailed descriptions)
- View extracted analysis from all attachments
- Alternative: Use `AttachmentFetcher` for integrated processing

## Part B: Local File Testing
- Create test images with PIL
- **Test multimodal image analysis endpoint** (new primary method)
- Compare OCR vs Image Analysis results
- Test document recognition

## Part C: Parallel Processing Testing
- Compare sequential vs parallel processing times
- Test AttachmentFetcher parallel processing
- Test rate limiting functionality

### Key Changes from OCR to Multimodal Analysis

| Feature | OCR (`recognize_ocr`) | Multimodal (`analyze_image`) |
|---------|----------------------|------------------------------|
| Output | Text extraction only | Detailed descriptions |
| Diagrams | Extracts visible text | Describes components & relationships |
| Charts | Extracts labels/values | Explains trends & meaning |
| Images | Limited to text | Full visual understanding |
| Use Case | Text-heavy documents | Architecture diagrams, flowcharts |

### Parallel Processing Options

```python
from src.preprocessing import ParallelProcessor, ParallelConfig

# Direct parallel processing
processor = ParallelProcessor(max_workers=4, rate_limit_rps=10.0)
results = processor.map(client.analyze_image, image_paths)

# AttachmentFetcher parallel processing
results = fetcher.process_attachments_parallel(
    attachments=image_attachments,
    page_id=page_id,
    max_workers=4,
    rate_limit_rps=10.0,
)

# Full pipeline with parallel processing
from src.preprocessing import PreprocessingPipeline, ParallelConfig
pipeline = PreprocessingPipeline.from_env()
pipeline.process(
    "input.json",
    parallel=True,
    parallel_config=ParallelConfig(
        max_workers_pages=8,
        max_workers_attachments=4,
        rate_limit_rps=10.0,
        batch_size=50,
    )
)
```

### API Methods

```python
# New primary method for images (recommended)
analysis = client.analyze_image("diagram.png")
analysis = client.analyze_image("chart.png", prompt="What trends does this show?")

# Legacy OCR (still available for text extraction)
text = client.recognize_ocr("document_scan.png")

# High-level interface via TextRecognizer
recognizer = TextRecognizer(client)
analysis = recognizer.analyze_image("architecture.png")  # New method
text = recognizer.recognize_image("scan.png")  # Uses multimodal by default now
```

### Troubleshooting Guide

If image analysis returns empty or errors:
1. Verify the model supports vision (gpt-4o, claude-4.5-sonnet, etc.)
2. Check image format is supported (JPEG, PNG, WebP, GIF)
3. Ensure image file is not corrupted
4. Check API key has permissions for chat endpoint
5. Verify base64 encoding is working correctly

---
# Summary

## Part A: Confluence Page Testing
- Fetch a specific Confluence page by ID
- List all attachments on the page
- Filter for image attachments
- **Download and process each image with multimodal LLM analysis** (provides detailed descriptions)
- View extracted analysis from all attachments
- Alternative: Use `AttachmentFetcher` for integrated processing

## Part B: Local File Testing
- Create test images with PIL
- **Test multimodal image analysis endpoint** (new primary method)
- Compare OCR vs Image Analysis results
- Test document recognition

### Key Changes from OCR to Multimodal Analysis

| Feature | OCR (`recognize_ocr`) | Multimodal (`analyze_image`) |
|---------|----------------------|------------------------------|
| Output | Text extraction only | Detailed descriptions |
| Diagrams | Extracts visible text | Describes components & relationships |
| Charts | Extracts labels/values | Explains trends & meaning |
| Images | Limited to text | Full visual understanding |
| Use Case | Text-heavy documents | Architecture diagrams, flowcharts |

### API Methods

```python
# New primary method for images (recommended)
analysis = client.analyze_image("diagram.png")
analysis = client.analyze_image("chart.png", prompt="What trends does this show?")

# Legacy OCR (still available for text extraction)
text = client.recognize_ocr("document_scan.png")

# High-level interface via TextRecognizer
recognizer = TextRecognizer(client)
analysis = recognizer.analyze_image("architecture.png")  # New method
text = recognizer.recognize_image("scan.png")  # Uses multimodal by default now
```

### Troubleshooting Guide

If image analysis returns empty or errors:
1. Verify the model supports vision (gpt-4o, claude-4.5-sonnet, etc.)
2. Check image format is supported (JPEG, PNG, WebP, GIF)
3. Ensure image file is not corrupted
4. Check API key has permissions for chat endpoint
5. Verify base64 encoding is working correctly

In [ ]:
# Import parallel processing utilities
from src.preprocessing.parallel import ParallelProcessor, RateLimiter
import time

print("Parallel processing utilities loaded!")
print(f"  - ParallelProcessor: For concurrent task execution")
print(f"  - RateLimiter: Token bucket rate limiting")

## C1. Compare Sequential vs Parallel Processing

Process the same images sequentially and in parallel to measure speedup.

In [ ]:
# Compare sequential vs parallel processing
if image_attachments and len(image_attachments) >= 2:
    # Get downloaded image paths
    image_paths = [
        str(attachment_dir / att.get('title'))
        for att in image_attachments
        if (attachment_dir / att.get('title')).exists()
    ]
    
    if len(image_paths) >= 2:
        print(f"Testing with {len(image_paths)} images...")
        print("=" * 60)
        
        # Sequential processing
        print("\n📝 Sequential Processing:")
        print("-" * 40)
        seq_start = time.time()
        seq_results = []
        for path in image_paths:
            result = client.analyze_image(path)
            seq_results.append(result)
        seq_time = time.time() - seq_start
        print(f"   Time: {seq_time:.2f}s")
        print(f"   Results: {len(seq_results)} images processed")
        
        # Parallel processing
        print("\n⚡ Parallel Processing (4 workers):")
        print("-" * 40)
        processor = ParallelProcessor(max_workers=4)
        par_start = time.time()
        par_results = processor.map(client.analyze_image, image_paths, desc="Images")
        par_time = time.time() - par_start
        processor.shutdown()
        
        successful = sum(1 for r in par_results if r.success)
        print(f"   Time: {par_time:.2f}s")
        print(f"   Results: {successful}/{len(par_results)} successful")
        
        # Summary
        print("\n" + "=" * 60)
        print("📊 PERFORMANCE COMPARISON")
        print("=" * 60)
        print(f"   Sequential: {seq_time:.2f}s")
        print(f"   Parallel:   {par_time:.2f}s")
        speedup = seq_time / par_time if par_time > 0 else 0
        print(f"   Speedup:    {speedup:.1f}x")
        
        if speedup > 1:
            time_saved = seq_time - par_time
            print(f"\n   ✅ Parallel processing saved {time_saved:.1f} seconds!")
    else:
        print("Not enough downloaded images for comparison")
else:
    print("Need at least 2 image attachments for comparison testing")

## C2. Test AttachmentFetcher Parallel Processing

Use the AttachmentFetcher's built-in parallel processing for within-page parallelization.

In [ ]:
# Test AttachmentFetcher parallel processing
if confluence_client and CONFLUENCE_PAGE_ID and image_attachments:
    print("Testing AttachmentFetcher parallel processing...")
    print("=" * 60)
    
    fetcher = AttachmentFetcher(
        confluence_client=confluence_client,
        iliad_client=client,
        storage_path="test_files/parallel_test"
    )
    
    # Test parallel processing within a page
    print("\n⚡ Processing attachments in parallel (4 workers):")
    print("-" * 40)
    
    par_start = time.time()
    results = fetcher.process_attachments_parallel(
        attachments=image_attachments,
        page_id=CONFLUENCE_PAGE_ID,
        max_workers=4,
        rate_limit_rps=None,  # No rate limiting for this test
        cleanup=True,
    )
    par_time = time.time() - par_start
    
    successful = sum(1 for r in results if r.get('success'))
    print(f"\n✅ Results:")
    print(f"   Time: {par_time:.2f}s")
    print(f"   Successful: {successful}/{len(results)}")
    
    for r in results:
        status = "✅" if r.get('success') else "❌"
        text_len = len(r.get('extracted_text', ''))
        print(f"   {status} {r.get('filename')}: {text_len} chars")
else:
    print("⚠️  Cannot test - need confluence_client, page ID, and image attachments")

## C3. Test Rate Limiting

Test the rate limiter to ensure it properly throttles API calls.

In [ ]:
# Test rate limiter
print("Testing RateLimiter (5 requests per second)...")
print("-" * 60)

limiter = RateLimiter(requests_per_second=5.0)
test_count = 10

start_time = time.time()
acquire_times = []

for i in range(test_count):
    t0 = time.time()
    limiter.acquire()
    t1 = time.time()
    acquire_times.append(t1 - t0)
    print(f"  Request {i+1}: acquired in {acquire_times[-1]*1000:.1f}ms")

total_time = time.time() - start_time
expected_time = (test_count - 1) / 5.0  # First request is instant

print(f"\n📊 Rate Limiter Results:")
print(f"   Total requests: {test_count}")
print(f"   Total time: {total_time:.2f}s")
print(f"   Expected time: ~{expected_time:.2f}s")
print(f"   Effective RPS: {test_count / total_time:.1f}")

if total_time >= expected_time * 0.9:
    print(f"\n   ✅ Rate limiter is working correctly!")